In [5]:
# 서울시 싱크홀 위험도 분석 - 데이터 전처리 파이프라인
# Jupyter Notebook 버전

# %% [markdown]
"""
# 서울시 싱크홀 위험도 분석 데이터 전처리

이 노트북은 서울시의 포트홀, 싱크홀, 강수량 데이터를 통합하여 
Azure ML Studio에서 사용할 수 있는 학습 데이터셋을 생성합니다.

## 주요 기능
1. 다중 해상도 데이터 통합 (그리드 기반)
2. 복합 위험도 지표 생성
3. Azure ML Studio 연동 준비
"""

# %% [markdown] 
## 1. 라이브러리 설치 및 임포트

# %%
# 필요한 라이브러리 설치 (Colab/로컬 환경에서 필요시)
# !pip install pandas numpy folium geopy openpyxl

# %%
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import warnings
from geopy.distance import geodesic
import os
from typing import Tuple, Dict, List
import json

# 한글 폰트 설정 (matplotlib)
plt.rcParams['font.family'] = 'DejaVu Sans'
plt.rcParams['axes.unicode_minus'] = False

# 경고 메시지 무시
warnings.filterwarnings('ignore')

# 출력 옵션 설정
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

print("✅ 라이브러리 임포트 완료!")

# %% [markdown]
## 2. 데이터 전처리 클래스 정의

# %%
class SeoulRiskDataProcessor:
    """서울시 위험도 데이터 전처리 클래스 - Jupyter Notebook 버전"""
    
    def __init__(self, grid_precision: int = 3):
        self.grid_precision = grid_precision
        self.grid_size = 10 ** (-grid_precision)
        
        # 서울시 경계 설정
        self.seoul_bounds = {
            'lat_min': 37.42, 'lat_max': 37.70,
            'lon_min': 126.76, 'lon_max': 127.18
        }
        
        print(f"🌐 그리드 설정: {grid_precision}자리 정밀도 (약 {111 * self.grid_size:.0f}m 단위)")
        
    def normalize_coordinates(self, lat: float, lon: float) -> Tuple[float, float]:
        """좌표 정규화"""
        factor = 10 ** self.grid_precision
        return round(lat * factor) / factor, round(lon * factor) / factor
    
    def create_grid_id(self, lat: float, lon: float) -> str:
        """그리드 ID 생성"""
        norm_lat, norm_lon = self.normalize_coordinates(lat, lon)
        return f"{norm_lat}_{norm_lon}"
    
    def is_in_seoul(self, lat: float, lon: float) -> bool:
        """서울시 경계 확인"""
        return (self.seoul_bounds['lat_min'] <= lat <= self.seoul_bounds['lat_max'] and
                self.seoul_bounds['lon_min'] <= lon <= self.seoul_bounds['lon_max'])

# 프로세서 인스턴스 생성
processor = SeoulRiskDataProcessor(grid_precision=3)

# %% [markdown]
## 3. 데이터 로딩 및 기본 탐색

# %%
# 파일 경로 설정 (파일이 있는 경로로 수정하세요)
file_paths = {
    'porthole': 'porthole_0624_5.csv',
    'sinkhole': 'sinkhole_0624_4.csv', 
    'rainfall': 'rainfall_cleaned_utf8.csv',
    'repair': '서울시 포트홀 보수 위치 정보.xlsx'
}

# 데이터 로딩
print("📂 데이터 로딩 중...")
datasets = {}

# %%
# 포트홀 데이터 로딩
print("1️⃣ 포트홀 데이터 로딩")
porthole_df = pd.read_csv(file_paths['porthole'])

print(f"원본 데이터: {len(porthole_df):,}행")
print("컬럼:", list(porthole_df.columns))
print("\n샘플 데이터:")
display(porthole_df.head())

# 데이터 정제
porthole_df = porthole_df.dropna(subset=['사고지점위도', '사고지점경도'])
porthole_df = porthole_df[
    porthole_df.apply(lambda row: processor.is_in_seoul(row['사고지점위도'], row['사고지점경도']), axis=1)
]
porthole_df['사고발생일자'] = pd.to_datetime(porthole_df['사고발생일자'], format='%Y%m%d', errors='coerce')
porthole_df = porthole_df.dropna(subset=['사고발생일자'])

datasets['porthole'] = porthole_df
print(f"✅ 정제 후: {len(porthole_df):,}행 (서울시 범위 내)")

# %%
# 싱크홀 데이터 로딩
print("2️⃣ 싱크홀 데이터 로딩")
sinkhole_df = pd.read_csv(file_paths['sinkhole'])

print(f"원본 데이터: {len(sinkhole_df):,}행")
print("컬럼:", list(sinkhole_df.columns))
print("\n샘플 데이터:")
display(sinkhole_df.head())

# 데이터 정제
sinkhole_df = sinkhole_df.dropna(subset=['사고지점위도', '사고지점경도'])
sinkhole_df = sinkhole_df[
    sinkhole_df.apply(lambda row: processor.is_in_seoul(row['사고지점위도'], row['사고지점경도']), axis=1)
]
sinkhole_df['사고발생일자'] = pd.to_datetime(sinkhole_df['사고발생일자'], format='%Y%m%d', errors='coerce')
sinkhole_df = sinkhole_df.dropna(subset=['사고발생일자'])

datasets['sinkhole'] = sinkhole_df
print(f"✅ 정제 후: {len(sinkhole_df):,}행")

# %%
# 강수량 데이터 로딩
print("3️⃣ 강수량 데이터 로딩")
rainfall_df = pd.read_csv(file_paths['rainfall'])

print(f"원본 데이터: {len(rainfall_df):,}행")
print("컬럼:", list(rainfall_df.columns))
print("\n샘플 데이터:")
display(rainfall_df.head())

# 데이터 정제
rainfall_df['SAWS_OBS_TM'] = pd.to_datetime(rainfall_df['SAWS_OBS_TM'], errors='coerce')
rainfall_df = rainfall_df.dropna(subset=['SAWS_OBS_TM', 'SAWS_RN_SUM'])
rainfall_df = rainfall_df.dropna(subset=['위도', '경도'])
rainfall_df = rainfall_df[
    rainfall_df.apply(lambda row: processor.is_in_seoul(row['위도'], row['경도']), axis=1)
]

datasets['rainfall'] = rainfall_df
print(f"✅ 정제 후: {len(rainfall_df):,}행")

# %% [markdown]
## 4. 데이터 시각화 및 탐색적 분석

# %%
# 사고 데이터 기본 통계
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# 포트홀 사고 시간대별 분포
porthole_df['year'] = porthole_df['사고발생일자'].dt.year
porthole_df['month'] = porthole_df['사고발생일자'].dt.month

axes[0,0].hist(porthole_df['year'], bins=20, alpha=0.7, color='skyblue')
axes[0,0].set_title('포트홀 사고 연도별 분포')
axes[0,0].set_xlabel('연도')
axes[0,0].set_ylabel('사고 건수')

axes[0,1].hist(porthole_df['month'], bins=12, alpha=0.7, color='lightcoral')
axes[0,1].set_title('포트홀 사고 월별 분포')
axes[0,1].set_xlabel('월')
axes[0,1].set_ylabel('사고 건수')

# 싱크홀 규모 분포 (규모 정보가 있는 경우)
if '발생규모폭(m)' in sinkhole_df.columns:
    sinkhole_df['발생규모폭(m)'] = pd.to_numeric(sinkhole_df['발생규모폭(m)'], errors='coerce')
    axes[1,0].hist(sinkhole_df['발생규모폭(m)'].dropna(), bins=20, alpha=0.7, color='lightgreen')
    axes[1,0].set_title('싱크홀 규모(폭) 분포')
    axes[1,0].set_xlabel('폭 (m)')
    axes[1,0].set_ylabel('건수')

# 자치구별 강수량 평균
district_rainfall = rainfall_df.groupby('CGG_NM')['SAWS_RN_SUM'].mean().sort_values(ascending=False)
axes[1,1].bar(range(len(district_rainfall)), district_rainfall.values)
axes[1,1].set_title('자치구별 평균 강수량')
axes[1,1].set_xlabel('자치구')
axes[1,1].set_ylabel('평균 강수량 (mm)')
axes[1,1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

print("📊 기본 통계:")
print(f"포트홀 사고 기간: {porthole_df['사고발생일자'].min()} ~ {porthole_df['사고발생일자'].max()}")
print(f"싱크홀 사고 기간: {sinkhole_df['사고발생일자'].min()} ~ {sinkhole_df['사고발생일자'].max()}")
print(f"강수량 데이터 기간: {rainfall_df['SAWS_OBS_TM'].min()} ~ {rainfall_df['SAWS_OBS_TM'].max()}")

# %% [markdown]
## 5. 그리드 생성 및 데이터 집계

# %%
def create_seoul_grid(processor):
    """서울시 그리드 생성"""
    print("🗺️ 서울시 그리드 생성 중...")
    
    lat_range = np.arange(
        processor.seoul_bounds['lat_min'], 
        processor.seoul_bounds['lat_max'] + processor.grid_size, 
        processor.grid_size
    )
    lon_range = np.arange(
        processor.seoul_bounds['lon_min'], 
        processor.seoul_bounds['lon_max'] + processor.grid_size, 
        processor.grid_size
    )
    
    grid_data = []
    for lat in lat_range:
        for lon in lon_range:
            grid_id = processor.create_grid_id(lat, lon)
            grid_data.append({
                'grid_id': grid_id,
                'grid_lat': lat,
                'grid_lon': lon
            })
    
    grid_df = pd.DataFrame(grid_data)
    print(f"✅ 총 {len(grid_df):,}개 그리드 생성")
    return grid_df

# 그리드 생성
grid_df = create_seoul_grid(processor)
display(grid_df.head())

# %%
def aggregate_accident_data(grid_df, accidents_df, accident_type):
    """사고 데이터 그리드 집계"""
    print(f"📊 {accident_type} 데이터 집계 중...")
    
    # 그리드 ID 할당
    accidents_df['grid_id'] = accidents_df.apply(
        lambda row: processor.create_grid_id(row['사고지점위도'], row['사고지점경도']), 
        axis=1
    )
    
    # 시간 피처 추출
    accidents_df['year'] = accidents_df['사고발생일자'].dt.year
    accidents_df['month'] = accidents_df['사고발생일자'].dt.month
    accidents_df['is_rainy_season'] = accidents_df['month'].isin([6, 7, 8])
    
    # 그리드별 집계
    grid_stats = accidents_df.groupby('grid_id').agg({
        '사고발생일자': ['count', 'min', 'max'],
        'year': 'nunique',
        'is_rainy_season': 'sum'
    }).reset_index()
    
    # 컬럼명 정리
    grid_stats.columns = [
        'grid_id',
        f'{accident_type}_count',
        f'{accident_type}_first_date',
        f'{accident_type}_last_date',
        f'{accident_type}_years_active',
        f'{accident_type}_rainy_season_count'
    ]
    
    # 추가 피처
    current_date = datetime.now()
    grid_stats[f'{accident_type}_days_since_last'] = (
        current_date - grid_stats[f'{accident_type}_last_date']
    ).dt.days
    
    grid_stats[f'{accident_type}_density_per_year'] = (
        grid_stats[f'{accident_type}_count'] / 
        np.maximum(grid_stats[f'{accident_type}_years_active'], 1)
    )
    
    # 병합
    result_df = grid_df.merge(grid_stats, on='grid_id', how='left')
    accident_columns = [col for col in result_df.columns if accident_type in col and col != 'grid_id']
    result_df[accident_columns] = result_df[accident_columns].fillna(0)
    
    print(f"✅ {grid_stats[f'{accident_type}_count'].sum():,}건을 {len(grid_stats):,}개 그리드에 분배")
    return result_df

# 포트홀 데이터 집계
result_df = aggregate_accident_data(grid_df, datasets['porthole'], 'porthole')
display(result_df[result_df['porthole_count'] > 0].head())

# %%
# 싱크홀 데이터 집계
result_df = aggregate_accident_data(result_df, datasets['sinkhole'], 'sinkhole')
print(f"현재 피처 수: {len(result_df.columns)}개")

# %% [markdown]
## 6. 강수량 데이터 집계

# %%
def aggregate_rainfall_data(grid_df, rainfall_df):
    """강수량 데이터 그리드 집계"""
    print("🌧️ 강수량 데이터 집계 중...")
    
    # 자치구별 대표 좌표 매핑
    district_grid_mapping = {}
    for _, row in rainfall_df[['CGG_NM', '위도', '경도']].drop_duplicates().iterrows():
        grid_id = processor.create_grid_id(row['위도'], row['경도'])
        district_grid_mapping[row['CGG_NM']] = grid_id
    
    # 자치구별 강수량 통계
    district_stats = rainfall_df.groupby('CGG_NM').agg({
        'SAWS_RN_SUM': ['mean', 'std', 'max', 'count']
    }).reset_index()
    
    district_stats.columns = [
        'district', 'avg_daily_rainfall', 'std_daily_rainfall', 
        'max_daily_rainfall', 'total_observations'
    ]
    
    # 그리드 ID 매핑
    district_stats['grid_id'] = district_stats['district'].map(district_grid_mapping)
    district_stats = district_stats.dropna(subset=['grid_id'])
    
    # 병합
    result_df = grid_df.merge(
        district_stats.drop('district', axis=1), 
        on='grid_id', 
        how='left'
    )
    
    # 결측값 처리
    rainfall_columns = ['avg_daily_rainfall', 'std_daily_rainfall', 'max_daily_rainfall', 'total_observations']
    for col in rainfall_columns:
        if col in result_df.columns:
            result_df[col] = result_df[col].fillna(result_df[col].mean())
    
    print(f"✅ {len(district_stats)}개 자치구 강수량 데이터 매핑 완료")
    return result_df

# 강수량 데이터 집계
result_df = aggregate_rainfall_data(result_df, datasets['rainfall'])
print(f"현재 피처 수: {len(result_df.columns)}개")

# %% [markdown]
## 7. 복합 위험도 지표 생성

# %%
def create_composite_risk_features(df):
    """복합 위험도 지표 생성"""
    print("🎯 복합 위험도 지표 계산 중...")
    
    # 1. 사고 위험도 점수
    accident_risk_cols = [col for col in df.columns if 'count' in col and ('porthole' in col or 'sinkhole' in col)]
    if accident_risk_cols:
        df['historical_accident_score'] = df[accident_risk_cols].sum(axis=1)
    else:
        df['historical_accident_score'] = 0
    
    # 2. 강수량 위험도 점수 (장마철 8.4배 가중)
    if 'avg_daily_rainfall' in df.columns:
        df['rainfall_risk_score'] = df['avg_daily_rainfall'] * 8.4
    else:
        df['rainfall_risk_score'] = 0
    
    # 3. 계절성 위험도
    if 'porthole_rainy_season_count' in df.columns and 'sinkhole_rainy_season_count' in df.columns:
        df['seasonal_risk_score'] = (
            df['porthole_rainy_season_count'] + 
            df['sinkhole_rainy_season_count'] * 2
        )
    else:
        df['seasonal_risk_score'] = 0
    
    # 4. 최종 복합 위험도 점수
    df['composite_risk_score'] = (
        df['historical_accident_score'] * 0.5 +
        df['rainfall_risk_score'] * 0.3 +
        df['seasonal_risk_score'] * 0.2
    )
    
    # 5. 0-100 범위로 정규화
    if df['composite_risk_score'].max() > 0:
        df['composite_risk_score_normalized'] = (
            df['composite_risk_score'] / df['composite_risk_score'].max() * 100
        )
    else:
        df['composite_risk_score_normalized'] = 0
    
    # 6. 위험 등급 분류
    df['risk_level'] = pd.cut(
        df['composite_risk_score_normalized'],
        bins=[-1, 20, 40, 60, 80, 100],
        labels=['매우낮음', '낮음', '보통', '높음', '매우높음']
    )
    
    print("✅ 복합 위험도 지표 생성 완료")
    print("위험도 분포:")
    print(df['risk_level'].value_counts())
    
    return df

# 복합 위험도 지표 생성
final_df = create_composite_risk_features(result_df)

# %% [markdown]
## 8. 결과 확인 및 시각화

# %%
# 최종 데이터셋 정보
print("📋 최종 데이터셋 정보:")
print(f"총 그리드 수: {len(final_df):,}개")
print(f"총 피처 수: {len(final_df.columns):,}개")
print(f"사고가 발생한 그리드: {len(final_df[final_df['composite_risk_score'] > 0]):,}개")

# 주요 통계
print("\n📊 위험도 통계:")
print(final_df['composite_risk_score_normalized'].describe())

# 피처 목록
print("\n🔍 생성된 피처:")
feature_categories = {
    '기본': [col for col in final_df.columns if 'grid' in col],
    '포트홀': [col for col in final_df.columns if 'porthole' in col],
    '싱크홀': [col for col in final_df.columns if 'sinkhole' in col],
    '강수량': [col for col in final_df.columns if 'rainfall' in col],
    '복합지표': [col for col in final_df.columns if 'risk' in col or 'level' in col]
}

for category, features in feature_categories.items():
    print(f"{category}: {len(features)}개 - {features[:3]}...")

# %%
# 위험도 분포 시각화
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# 위험도 히스토그램
axes[0,0].hist(final_df['composite_risk_score_normalized'], bins=50, alpha=0.7, color='skyblue')
axes[0,0].set_title('위험도 점수 분포')
axes[0,0].set_xlabel('위험도 점수 (0-100)')
axes[0,0].set_ylabel('그리드 수')

# 위험 등급별 분포
risk_counts = final_df['risk_level'].value_counts()
axes[0,1].pie(risk_counts.values, labels=risk_counts.index, autopct='%1.1f%%')
axes[0,1].set_title('위험 등급 분포')

# 상위 위험 지역 산점도
high_risk = final_df[final_df['composite_risk_score_normalized'] > 80]
axes[1,0].scatter(high_risk['grid_lon'], high_risk['grid_lat'], 
                 c=high_risk['composite_risk_score_normalized'], 
                 cmap='Reds', alpha=0.7)
axes[1,0].set_title('고위험 지역 분포 (위험도 80+ 지역)')
axes[1,0].set_xlabel('경도')
axes[1,0].set_ylabel('위도')

# 사고 건수 vs 위험도 상관관계
total_accidents = final_df['porthole_count'] + final_df['sinkhole_count']
axes[1,1].scatter(total_accidents, final_df['composite_risk_score_normalized'], alpha=0.5)
axes[1,1].set_title('총 사고 건수 vs 위험도')
axes[1,1].set_xlabel('총 사고 건수')
axes[1,1].set_ylabel('위험도 점수')

plt.tight_layout()
plt.show()

# %% [markdown]
## 9. Azure ML Studio 준비

# %%
# Azure ML Studio용 데이터 준비
def prepare_for_azure_ml(df, output_dir='processed_data'):
    """Azure ML Studio용 데이터 준비"""
    os.makedirs(output_dir, exist_ok=True)
    
    # 1. 메인 데이터셋 저장
    main_file = os.path.join(output_dir, 'seoul_risk_dataset.csv')
    df.to_csv(main_file, index=False, encoding='utf-8-sig')
    print(f"✅ 메인 데이터셋 저장: {main_file}")
    
    # 2. 피처 정보 저장
    feature_info = {
        'total_features': len(df.columns),
        'total_grids': len(df),
        'target_variable': 'composite_risk_score_normalized',
        'feature_categories': feature_categories,
        'risk_distribution': df['risk_level'].value_counts().to_dict()
    }
    
    info_file = os.path.join(output_dir, 'feature_info.json')
    with open(info_file, 'w', encoding='utf-8') as f:
        json.dump(feature_info, f, ensure_ascii=False, indent=2, default=str)
    print(f"✅ 피처 정보 저장: {info_file}")
    
    # 3. 샘플 데이터 저장
    sample_file = os.path.join(output_dir, 'sample_data.csv')
    df.sample(min(1000, len(df))).to_csv(sample_file, index=False, encoding='utf-8-sig')
    print(f"✅ 샘플 데이터 저장: {sample_file}")
    
    return main_file, feature_info

# Azure ML 준비
azure_file, feature_info = prepare_for_azure_ml(final_df)

# %%
# Azure ML Studio 설정 안내
print("🚀 Azure ML Studio 사용 가이드")
print("=" * 50)
print(f"1. 업로드할 파일: {azure_file}")
print(f"2. 타겟 변수: composite_risk_score_normalized")
print(f"3. 작업 유형: 회귀 (Regression)")
print(f"4. 기본 메트릭: normalized_mean_absolute_error")
print(f"5. 교차 검증: 5-fold")

print("\n📊 데이터셋 요약:")
print(f"- 총 그리드: {len(final_df):,}개")
print(f"- 총 피처: {len(final_df.columns):,}개") 
print(f"- 위험도 범위: 0-100")
print(f"- 고위험 지역: {len(final_df[final_df['composite_risk_score_normalized'] > 60]):,}개")

print("\n🎯 기대 성능:")
print("- 기본 정확도: 70-80% (NMAE)")
print("- 최적화 후: 85-90%")

# %%
# 최종 데이터 확인
print("📋 최종 데이터셋 미리보기:")
display(final_df.head(10))

print("\n🔍 고위험 지역 Top 10:")
high_risk_areas = final_df.nlargest(10, 'composite_risk_score_normalized')[
    ['grid_id', 'grid_lat', 'grid_lon', 'composite_risk_score_normalized', 'risk_level',
     'porthole_count', 'sinkhole_count', 'avg_daily_rainfall']
]
display(high_risk_areas)

print("\n✅ 전처리 파이프라인 완료!")
print("이제 Azure ML Studio에서 모델 학습을 시작할 수 있습니다! 🎉")

# %%
# Azure ML Studio 업로드용 CSV 파일 저장
def save_for_azure_ml(df, output_dir='azure_ml_ready'):
    """Azure ML Studio 업로드를 위한 CSV 파일 저장"""
    print("=== Azure ML Studio 업로드용 파일 저장 ===")
    
    # 출력 디렉토리 생성
    os.makedirs(output_dir, exist_ok=True)
    
    # 1. 메인 학습 데이터셋 저장
    main_file = os.path.join(output_dir, 'seoul_risk_dataset.csv')
    df.to_csv(main_file, index=False, encoding='utf-8-sig')
    print(f"✅ 메인 데이터셋: {main_file}")
    print(f"   - 행 수: {len(df):,}")
    print(f"   - 열 수: {len(df.columns):,}")
    print(f"   - 파일 크기: {os.path.getsize(main_file) / 1024 / 1024:.1f} MB")
    
    # 2. 피처 정보 저장 (JSON)
    feature_info = {
        'total_features': len(df.columns),
        'total_grids': len(df),
        'target_variable': 'composite_risk_score_normalized',
        'feature_list': list(df.columns),
        'risk_distribution': df['risk_level'].value_counts().to_dict() if 'risk_level' in df.columns else {},
        'data_summary': df.describe().to_dict()
    }
    
    info_file = os.path.join(output_dir, 'feature_info.json')
    with open(info_file, 'w', encoding='utf-8') as f:
        json.dump(feature_info, f, ensure_ascii=False, indent=2, default=str)
    print(f"✅ 피처 정보: {info_file}")
    
    # 3. 샘플 데이터 저장 (확인용)
    sample_file = os.path.join(output_dir, 'sample_preview.csv')
    sample_size = min(100, len(df))
    df.head(sample_size).to_csv(sample_file, index=False, encoding='utf-8-sig')
    print(f"✅ 샘플 데이터: {sample_file} ({sample_size}행)")
    
    # 4. Azure ML 업로드 가이드 저장
    guide_content = f"""# Azure ML Studio 업로드 가이드

## 📁 업로드할 파일
- **파일명**: seoul_risk_dataset.csv
- **크기**: {os.path.getsize(main_file) / 1024 / 1024:.1f} MB
- **행 수**: {len(df):,}
- **열 수**: {len(df.columns):,}

## 🎯 Azure ML Studio 설정
1. **데이터셋 업로드**
   - Azure ML Studio > 데이터 > 데이터셋 > 파일에서 만들기
   - 파일 선택: seoul_risk_dataset.csv
   - 이름: seoul_sinkhole_risk_data

2. **Designer 파이프라인 설정**
   - 작업 유형: 회귀 (Regression)
   - 타겟 열: composite_risk_score_normalized
   - 추천 모델: Decision Forest Regression, Boosted Decision Tree Regression

3. **피처 설명**
   - 좌표: grid_lat, grid_lon
   - 사고이력: porthole_count, sinkhole_count
   - 환경: avg_daily_rainfall, max_daily_rainfall
   - 타겟: composite_risk_score_normalized (0-100)

## 📊 예상 성능
- RMSE: 12-18
- R²: 0.80-0.90
- 학습시간: 30분-1시간
"""
    
    guide_file = os.path.join(output_dir, 'AZURE_ML_UPLOAD_GUIDE.md')
    with open(guide_file, 'w', encoding='utf-8') as f:
        f.write(guide_content)
    print(f"✅ 업로드 가이드: {guide_file}")
    
    print(f"\n🚀 준비 완료!")
    print(f"📁 폴더: {os.path.abspath(output_dir)}")
    print(f"💡 다음 단계: Azure ML Studio에서 seoul_risk_dataset.csv 업로드")
    
    return main_file, feature_info

# 최종 데이터 저장 실행
azure_file, feature_info = save_for_azure_ml(final_df)

# %%
# 저장된 파일 확인
print("=== 저장된 파일 확인 ===")
import glob

output_files = glob.glob('azure_ml_ready/*')
for file_path in output_files:
    file_size = os.path.getsize(file_path) / 1024  # KB
    print(f"📄 {os.path.basename(file_path)}: {file_size:.1f} KB")

print(f"\n📊 데이터셋 요약:")
print(f"- 총 그리드: {len(final_df):,}개")
print(f"- 사고 발생 그리드: {len(final_df[final_df['composite_risk_score'] > 0]):,}개")
print(f"- 고위험 그리드: {len(final_df[final_df['composite_risk_score_normalized'] > 60]):,}개")

# %%
# Azure ML Studio 업로드 체크리스트
print("=== Azure ML Studio 업로드 체크리스트 ===")
checklist = [
    "✅ seoul_risk_dataset.csv 파일 저장 완료",
    "✅ 타겟 변수 'composite_risk_score_normalized' 확인",
    "✅ 피처들이 모두 숫자형(numeric) 타입인지 확인",
    "✅ 결측값(NaN) 처리 완료",
    "✅ 파일 크기가 적절한지 확인 (< 100MB 권장)",
]

for item in checklist:
    print(item)

print(f"\n🎯 다음 작업:")
print(f"1. azure_ml_ready/seoul_risk_dataset.csv 파일을 Azure ML Studio에 업로드")
print(f"2. Designer에서 Decision Forest Regression 파이프라인 구성")
print(f"3. 타겟 변수를 'composite_risk_score_normalized'로 설정")
print(f"4. 모델 학습 및 성능 평가")

# %% [markdown]
"""
## ✅ 완료된 작업

1. **데이터 전처리 완료**
   - 그리드 기반 데이터 통합
   - 복합 위험도 지표 생성
   - Azure ML 호환 형식으로 저장

2. **Azure ML Studio 준비 완료**
   - seoul_risk_dataset.csv 업로드 준비
   - 타겟 변수: composite_risk_score_normalized
   - 피처 정보 문서화

3. **다음 단계: Azure ML Studio 업로드**
   - Designer 파이프라인 구성
   - 모델 학습 및 평가
   - 웹 서비스 배포
"""

✅ 라이브러리 임포트 완료!
🌐 그리드 설정: 3자리 정밀도 (약 0m 단위)
📂 데이터 로딩 중...
1️⃣ 포트홀 데이터 로딩


FileNotFoundError: [Errno 2] No such file or directory: 'porthole_0624_5.csv'